# 10d -- CytoTRACE2 细胞分化潜能评估 (独立 Notebook)

本 notebook 对上皮谱系做 **CytoTRACE2 发育潜能分析**，是拟时序方法系列的第三个独立 notebook。
与 `10_pseudotime.ipynb`（综合 notebook：熵 + CytoTRACE v1 + Monocle3）和
`10b_pseudotime_monocle3.ipynb`（Monocle3 独立）并行存在，PI 可分别跑、分别看、最后横向比较
不同方法的 pseudotime / potency 结果。

## 方法原理

**CytoTRACE2**（Newman 实验室, 2024, `digitalcytometry/cytotrace2`）是第二代基于深度学习的
细胞分化潜能评估方法，与 CytoTRACE v1（`10_pseudotime.ipynb` 中 `CytoTRACEKernel` 使用的版本）
**完全不同**：

| 维度 | CytoTRACE v1 | CytoTRACE2（本 notebook）|
|------|-------------|--------------------------|
| 原理 | 基因计数与转录多样性（gene counts / GCS）| Gene Set Binarization + 预训练神经网络 |
| 输入 | 基因×细胞计数矩阵（算法内计算）| 基因×细胞表达矩阵（文件路径输入）|
| 起点 | 需要指定 root（起点）| **不需要**——输出绝对分化潜能等级 |
| 输出 | 连续分数（0-1，1=高潜能）+ 相对排序 | 连续分数 + 6 级分类 + 相对排序 |
| 运行方式 | 纯 Python（cellrank `CytoTRACEKernel`）| 预训练 DL 模型（需下载权重 ~数百 MB）|

CytoTRACE2 的 6 级分化潜能分类：
1. **Totipotent**（全能）— 可分化成完整生物体所有细胞类型
2. **Pluripotent**（多能）— 可分化为三胚层各谱系
3. **Multipotent**（多向）— 可分化为同一谱系内多种细胞类型
4. **Oligopotent**（寡向）— 可分化为少数几种紧密相关类型
5. **Unipotent**（单向）— 仅生成一种终末细胞类型
6. **Differentiated**（终末分化）— 不再分裂分化

## 输入与输出

| 项目 | 路径/字段 |
|------|-----------|
| 上游输入 | `UPSTREAM_PATH`（默认 `results/06_annotated_v1.h5ad`）|
| 输出 h5ad | `OUTPUT_PATH`（默认 `results/10d_pseudotime_cytotrace2_v1.h5ad`）|
| 新增 obs 列 | `cytotrace2_potency_score`（连续分数 0-1）、`cytotrace2_potency_category`（6 级分类）、`cytotrace2_relative_order` |
| 临时目录 | `results/_cytotrace2_10d_tmp/`（表达式导出 + CytoTRACE2 输出）|

## 与其他拟时序 notebook 的关系

| Notebook | 方法 | 产物 obs 列 |
|----------|------|-------------|
| `10_pseudotime.ipynb` | 综合（熵 + CytoTRACE v1 + Monocle3）| `entropy` / `cytotrace_score` / `pseudotime_monocle3_v1` |
| `10b_pseudotime_monocle3.ipynb` | Monocle3 独立 | `pseudotime_monocle3_v1` |
| `10c_pseudotime_cellrank.ipynb`（计划中）| CellRank | `pseudotime_cellrank_v1` |
| `10d_pseudotime_cytotrace2.ipynb`（本 notebook）| CytoTRACE2 | `cytotrace2_potency_score` / `cytotrace2_potency_category` / `cytotrace2_relative_order` |

## 依赖与环境

CytoTRACE2 Python 包（`cytotrace2-py`）因 numpy 版本冲突已隔离到
独立 conda 环境 `scrna-cytotrace2`（仿照 scCODA/scrna-sccoda 模式）。

安装：
```bash
conda env create -f environment-cytotrace2.yml
conda activate scrna-cytotrace2
```

使用时需切换到该环境 kernel 或通过 subprocess 调用。

> **重要提示**：本 notebook 中 CytoTRACE2 的 API 调用细节按官方 README 最佳猜测编写（当前环境未安装
> cytotrace2-py，无法实跑验证）。首次启用时如遇参数名/返回格式差异，请对照官方 README 微调：
> https://github.com/digitalcytometry/cytotrace2

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：06（细胞注释），读 `06_annotated_v*.h5ad`
- **同级**：`10_pseudotime.ipynb`（综合拟时序）、`10b_pseudotime_monocle3.ipynb`（Monocle3 独立）
- **下游**：本 stage 产出 checkpoint 供 16_trajectory_de 或后续可视化使用

### 为什么要迭代回跑？

CytoTRACE2 的输出质量取决于几个关键参数：
- **上皮筛选范围**（`EPITHELIAL_CLUSTERS` 或 `AUTO_SUBSET_EPITHELIAL`）：
  CytoTRACE2 的预训练模型在单一谱系上效果最佳。跨谱系混合会稀释分化潜能信号
- **输入计数矩阵的选择**（counts layer vs raw.X）：影响 Gene Set Binarization 的二值化质量
- **批次大小**（`CYTOTRACE2_BATCH_SIZE` / `CYTOTRACE2_SMOOTH_BATCH_SIZE`）：
  大样本可能需要调整以平衡速度与内存

CytoTRACE2 **不需要指定 root**（起点）——它输出绝对潜能，不依赖 pseudotime 方向。

### 如何回跑（三步操作）
1. 改 `UPSTREAM_PATH`——指向要复用的上游文件版本
2. 改 `OUTPUT_PATH`——bump 版本号 `_v1` -> `_v2`
3. 调整参数（在下方 `# === PARAMS ===` 区域）-> 重跑本 notebook（Cell -> Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。旧版 `.h5ad` 文件不覆盖不删除
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）
- **`promoted`**：PI 审查后确认可传给下游使用的正式版本
- **下游取数**：后续 stage 的 `UPSTREAM_PATH` 指向你决定采用的版本即可

### 追溯链（自动写入 h5ad 的 `adata.uns`）
- `stage` = `"10d_pseudotime_cytotrace2"`
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号
- `10d_pseudotime_cytotrace2_v1` = 方法参数嵌套 dict

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH              -- 06 注释结果 h5ad
# OUTPUT_PATH                -- 本 notebook 产出 checkpoint
# RUN_CYTOTRACE2             -- 布尔开关：PI 可关闭整段 CytoTRACE2
# CYTOTRACE2_SPECIES         -- 物种（"human" / "mouse"）
# CYTOTRACE2_WORK_DIR        -- 临时工作目录（表达式导出 + CytoTRACE2 输出）
# CYTOTRACE2_BATCH_SIZE      -- 每批处理细胞数（影响内存与速度）
# CYTOTRACE2_SMOOTH_BATCH_SIZE -- KNN 平滑批次大小
# CYTOTRACE2_MAX_WORKERS     -- 最大并行线程数
# CYTOTRACE2_DISABLE_PARALLEL -- 是否禁用并行化（调试用，默认 False）
# CLUSTER_KEY                -- 用于按簇聚合的 obs 列
# CELL_TYPE_COL              -- 优先使用的细胞类型列
# EPITHELIAL_CLUSTERS        -- 上皮谱系 cluster ID 列表；None=全体细胞
# AUTO_SUBSET_EPITHELIAL     -- 自动检测并筛选上皮细胞
# EPITHELIAL_LABEL_COL       -- 从哪个列检测上皮标签
# EPITHELIAL_KEYWORDS        -- 上皮关键词列表（小写匹配）

UPSTREAM_PATH = "results/06_annotated_v1.h5ad"
OUTPUT_PATH   = "results/10d_pseudotime_cytotrace2_v1.h5ad"

# === CytoTRACE2 开关 ===
RUN_CYTOTRACE2 = True  # PI 可设为 False 跳过整段 CytoTRACE2

# === CytoTRACE2 参数 ===
CYTOTRACE2_SPECIES = "human"
CYTOTRACE2_WORK_DIR = "results/_cytotrace2_10d_tmp"

# batch_size: 每批送入神经网络的细胞数。默认 10000；内存受限时降至 5000
CYTOTRACE2_BATCH_SIZE = 10000

# smooth_batch_size: KNN 平滑时的批次大小。默认 1000；大样本可增大以提速
CYTOTRACE2_SMOOTH_BATCH_SIZE = 1000

# max_workers: 并行数据加载线程数。默认 1；多核服务器可设 4-8
CYTOTRACE2_MAX_WORKERS = 1

# disable_parallelization: 完全禁用并行化（调试用，默认 False）
CYTOTRACE2_DISABLE_PARALLEL = False

# === 细胞类型 ===
CLUSTER_KEY   = "leiden_res_0.6"
CELL_TYPE_COL = "cell_type_final_v1"

# === 上皮筛选（CytoTRACE2 在单一谱系上效果最佳）===
EPITHELIAL_CLUSTERS = None  # None=全体细胞；PI 按需设如 ["0", "3", "5"]

AUTO_SUBSET_EPITHELIAL = True
EPITHELIAL_LABEL_COL = "cell_type_final_v1"
EPITHELIAL_KEYWORDS = ["epithelial", "parietal", "chief", "mucous", "foveolar",
                       "pit", "neck", "spem", "im_", "intestinal", "goblet"]

In [ ]:
# === setup：sys.path + env_check + 导入 + 切换目录 ===
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/07_downstream/ 回退两级，最后用 notebook 自身路径推算。
import sys, os, gc

_root = os.getcwd()
_root_candidates = [
    _root,
    os.path.abspath(os.path.join(_root, "..")),
    os.path.abspath(os.path.join(_root, "..", "..")),
]

for _cand in _root_candidates:
    if os.path.isdir(os.path.join(_cand, "src", "scrna_integration")):
        _root = _cand
        break
else:
    _root = os.environ.get("PROJECT_ROOT", _root)

if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs(CYTOTRACE2_WORK_DIR, exist_ok=True)
print(f"PROJECT_ROOT: {_root}")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")

# 环境自检（与其他 notebook 一致的 env_check 模式）
try:
    from scrna_integration.platform import env_check
    env_check(expected_env="scrna-integration")
except Exception as _e:
    print(f"环境自检跳过（env_check 不可用: {_e}）")

# 导入依赖
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import warnings

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  numpy {np.__version__}")

In [ ]:
# === 加载上游 h5ad ===
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")

# 确定实际使用的分组列
if CELL_TYPE_COL in adata.obs.columns:
    _group_col = CELL_TYPE_COL
    print(f"使用细胞类型列: {CELL_TYPE_COL}")
    _ct = adata.obs[CELL_TYPE_COL].dropna().astype(str)
    print(f"  细胞类型: {sorted(_ct.unique())}")
elif CLUSTER_KEY in adata.obs.columns:
    _group_col = CLUSTER_KEY
    print(f"CELL_TYPE_COL 不存在，fallback 到 CLUSTER_KEY: {CLUSTER_KEY}")
    print(f"  簇数: {adata.obs[CLUSTER_KEY].nunique()}")
else:
    raise KeyError(
        f"CELL_TYPE_COL '{CELL_TYPE_COL}' 和 "
        f"CLUSTER_KEY '{CLUSTER_KEY}' 都不在 obs 列中"
    )

# 检查 embedding
_has_umap = "X_umap" in adata.obsm
print(f"X_umap: {_has_umap}  |  obsm keys: {list(adata.obsm.keys())}")

In [ ]:
# === 上皮谱系筛选 ===
# CytoTRACE2 的预训练模型在单一谱系上效果最佳；跨谱系混合会稀释分化潜能信号。
# 提供两条路径（互斥）：
#   路径 1：手动指定 EPITHELIAL_CLUSTERS（如 ["0", "3", "5"]）
#   路径 2：AUTO_SUBSET_EPITHELIAL=True 自动关键词检测

# -- 路径 1：手动指定 --
if EPITHELIAL_CLUSTERS is not None:
    _epi_mask = adata.obs[_group_col].astype(str).isin(
        [str(c) for c in EPITHELIAL_CLUSTERS]
    )
    _n_before = adata.n_obs
    if not _epi_mask.any():
        raise ValueError(
            f"EPITHELIAL_CLUSTERS={EPITHELIAL_CLUSTERS} 不匹配任何细胞。"
            f"可用的 {_group_col} 值: "
            f"{sorted(adata.obs[_group_col].dropna().astype(str).unique())}"
        )
    adata = adata[_epi_mask].copy()
    print(
        f"上皮谱系筛选: {_n_before:,} -> {adata.n_obs:,} 细胞 "
        f"（保留 {_group_col}: {EPITHELIAL_CLUSTERS}）"
    )
else:
    print(
        "EPITHELIAL_CLUSTERS=None，对所有细胞做 CytoTRACE2 分析。"
        "PI 可在完成细胞类型注释后指定上皮 cluster 重新分析。"
    )

# -- 路径 2：自动关键词检测（仅在未手动指定时生效）--
if AUTO_SUBSET_EPITHELIAL and EPITHELIAL_CLUSTERS is None:
    _original_n = adata.n_obs
    if EPITHELIAL_LABEL_COL in adata.obs.columns:
        _labels = adata.obs[EPITHELIAL_LABEL_COL].astype(str).str.lower()
        _epi_mask = _labels.apply(lambda x: any(kw in x for kw in EPITHELIAL_KEYWORDS))

        if _epi_mask.sum() > 50:
            print(f"上皮自动筛选: {_epi_mask.sum():,}/{_original_n:,} cells 匹配上皮关键词")
            adata = adata[_epi_mask].copy()
            print(f"  筛选后: {adata.n_obs:,} cells")
            _top_labels = adata.obs[EPITHELIAL_LABEL_COL].value_counts().head(10)
            print(f"  包含标签: {_top_labels.to_dict()}")
        else:
            print(f"WARNING: 仅 {_epi_mask.sum()} cells 匹配上皮关键词（< 50），跳过筛选")
            print(f"  -> 检查 EPITHELIAL_KEYWORDS 或 EPITHELIAL_LABEL_COL 是否正确")
            AUTO_SUBSET_EPITHELIAL = False
    else:
        print(f"WARNING: {EPITHELIAL_LABEL_COL} 列不存在，跳过上皮筛选")
        AUTO_SUBSET_EPITHELIAL = False
elif not AUTO_SUBSET_EPITHELIAL and EPITHELIAL_CLUSTERS is None:
    print("AUTO_SUBSET_EPITHELIAL=False 且 EPITHELIAL_CLUSTERS=None，使用全部细胞")
elif EPITHELIAL_CLUSTERS is not None:
    print(f"EPITHELIAL_CLUSTERS 已手动指定 ({EPITHELIAL_CLUSTERS})，跳过自动筛选")

In [ ]:
# === CytoTRACE2 可用性守卫 ===
# 两层检测：
#   1. cytotrace2 包是否可 import（两种 import 路径均尝试）
#   2. 预训练权重/模型文件是否就绪（首次运行需联网下载 ~数百 MB）
# 任一层不满足 -> 优雅跳过，不崩 notebook

if RUN_CYTOTRACE2:
    # Check if we're in the correct conda environment
    import os
    _cytotrace2_env = "scrna-cytotrace2"
    _current_env = os.environ.get("CONDA_DEFAULT_ENV", "")
    if _current_env != _cytotrace2_env:
        print(f"⚠️ 当前 conda 环境是 '{_current_env}'，CytoTRACE2 需要 '{_cytotrace2_env}'")
        print(f"   请切换: conda activate {_cytotrace2_env}")

    _cytotrace2_available = False
    _cytotrace2_import_path = None
    _cytotrace2_module = None

    # 尝试两种 import 路径（以官方 README 为准，因无法实跑先后尝试两种）
    try:
        from cytotrace2_py.cytotrace2_py import cytotrace2 as _m
        _cytotrace2_available = True
        _cytotrace2_import_path = "cytotrace2_py.cytotrace2_py"
        _cytotrace2_module = _m
        print(f"cytotrace2 包可用（import 路径: cytotrace2_py.cytotrace2_py）")
    except ImportError as _e1:
        print(f"  import 路径 'cytotrace2_py.cytotrace2_py' 失败: {_e1}")
        try:
            import cytotrace2 as _m
            _cytotrace2_available = True
            _cytotrace2_import_path = "cytotrace2"
            _cytotrace2_module = _m
            print(f"cytotrace2 包可用（import 路径: cytotrace2）")
        except ImportError as _e2:
            print(f"  import 路径 'cytotrace2' 失败: {_e2}")

    if not _cytotrace2_available:
        print()
        print("=" * 60)
        print("cytotrace2 包不可用——所有 import 路径均失败。")
        print()
        print("安装方法:")
        print('  conda env create -f environment-cytotrace2.yml')
        print('  conda activate scrna-cytotrace2')
        print()
        print("本 notebook 将优雅跳过 CytoTRACE2 分析。")
        print("=" * 60)
        _cytotrace2_ready = False
    else:
        # 检测权重/模型文件是否就绪
        import pathlib

        _weight_checked = False
        _weights_found = False

        # 常见缓存位置
        _weight_locations = [
            pathlib.Path.home() / ".cytotrace2",
            pathlib.Path.home() / ".cache" / "cytotrace2",
        ]

        # 包内 models 目录（如果权重随包分发）
        try:
            _pkg_dir = pathlib.Path(_cytotrace2_module.__file__).parent
            _weight_locations.append(_pkg_dir / "models")
        except Exception:
            pass

        for _d in _weight_locations:
            if _d.exists():
                _contents = list(_d.iterdir())
                if _contents:
                    _weights_found = True
                    _weight_checked = True
                    print(f"权重/模型目录已就绪: {_d}  ({len(_contents)} 文件)")
                    break
                else:
                    print(f"权重目录存在但为空: {_d}")
                    _weight_checked = True

        if _weights_found:
            _cytotrace2_ready = True
            print("CytoTRACE2 可用性: 包 OK  权重 OK  -> 就绪")
        else:
            _cytotrace2_ready = False
            print()
            print("=" * 60)
            print("CytoTRACE2 权重未就绪！")
            print(f"  已检查位置: {[str(d) for d in _weight_locations]}")
            print()
            if not _weight_checked:
                print("  未检测到任何权重缓存目录。")
            print("  首次运行 cytotrace2() 会自动下载预训练权重（~数百 MB），")
            print("  请在有网络的环境执行一次即完成缓存。")
            print()
            print("  本 notebook 将优雅跳过 CytoTRACE2 分析。")
            print("=" * 60)
else:
    _cytotrace2_ready = False
    _cytotrace2_available = False
    print("RUN_CYTOTRACE2=False，CytoTRACE2 整段跳过")

In [ ]:
# === CytoTRACE2 运行：导出表达式 -> 调 cytotrace2() -> 获取结果 ===
# CytoTRACE2 Python 版接受文件路径（基因 x 细胞 TSV/CSV），不直接吃 AnnData。
# 流程：
#   1. 从 adata 提取计数矩阵（优先 counts layer > raw > X）
#   2. 导出为基因 x 细胞 TSV
#   3. 调 cytotrace2() 返回 AnnData（或 DataFrame）
#   4. 结果传给下一个 cell 写回 adata.obs
#
# 注意：本段 API 调用按官方 README 最佳猜测编写，因当前环境未安装 cytotrace2-py
# 无法实跑验证。首次启用时如遇参数名/返回格式差异，请对照官方 README 微调：
# https://github.com/digitalcytometry/cytotrace2

if RUN_CYTOTRACE2 and _cytotrace2_ready:
    # 清理并重建临时工作目录
    shutil.rmtree(CYTOTRACE2_WORK_DIR, ignore_errors=True)
    os.makedirs(CYTOTRACE2_WORK_DIR, exist_ok=True)

    # --- 获取计数矩阵 ---
    # CytoTRACE2 内部有 Gene Set Binarization 步骤，需要原始计数（非 log-normalized）
    if "counts" in adata.layers:
        _X_export = adata.layers["counts"]
        print("使用 adata.layers['counts'] 作为输入")
    elif adata.raw is not None:
        _X_export = adata.raw[:, adata.var_names].X
        print("使用 adata.raw.X 作为输入")
    else:
        _X_export = adata.X
        print("WARNING: 未找到 counts layer 或 raw，使用 adata.X（可能已 log-normalized）")

    # --- 导出基因 x 细胞 TSV ---
    # 格式：行=基因（第一列 GENE），列=细胞 barcode
    _tsv_path = os.path.join(CYTOTRACE2_WORK_DIR, "expr_counts.tsv")
    _X_dense = _X_export.toarray() if sp.issparse(_X_export) else np.asarray(_X_export)
    _expr_df = pd.DataFrame(
        _X_dense,
        index=adata.var_names.astype(str),
        columns=adata.obs_names.astype(str),
    )
    _expr_df.index.name = "GENE"
    _expr_df.to_csv(_tsv_path, sep="\t")
    print(f"\n表达矩阵已导出: {_tsv_path}")
    print(f"  维度: {_expr_df.shape[0]} genes x {_expr_df.shape[1]} cells")
    print(f"  文件大小: {os.path.getsize(_tsv_path):,} bytes")

    # --- 调用 CytoTRACE2 ---
    print(f"\n正在运行 CytoTRACE2...")
    print(f"  species={CYTOTRACE2_SPECIES}  batch_size={CYTOTRACE2_BATCH_SIZE}")
    print(f"  smooth_batch_size={CYTOTRACE2_SMOOTH_BATCH_SIZE}")
    print(f"  注意：首次运行需联网下载预训练权重，请确保网络畅通。")

    try:
        # API 调用——参数按官方 README 最佳猜测，首次启用时可能需要调整
        _ct2_result = _cytotrace2_module.cytotrace2(
            _tsv_path,
            species=CYTOTRACE2_SPECIES,
            batch_size=CYTOTRACE2_BATCH_SIZE,
            smooth_batch_size=CYTOTRACE2_SMOOTH_BATCH_SIZE,
            disable_parallelization=CYTOTRACE2_DISABLE_PARALLEL,
            max_workers=CYTOTRACE2_MAX_WORKERS,
        )
        print(f"\nCytoTRACE2 运行成功！")
        print(f"  返回类型: {type(_ct2_result).__name__}")
        if hasattr(_ct2_result, "obs"):
            print(f"  返回对象 obs 列: {list(_ct2_result.obs.columns)}")
            print(f"  细胞数: {_ct2_result.n_obs}")
        elif hasattr(_ct2_result, "shape"):
            print(f"  返回对象 shape: {_ct2_result.shape}")
        _cytotrace2_success = True
    except Exception as _e:
        print(f"\nCytoTRACE2 运行失败: {type(_e).__name__}: {_e}")
        print()
        print("=" * 60)
        print("注意：本 notebook 中的 CytoTRACE2 API 调用未经过实跑验证。")
        print("如遇参数错误，请对照官方文档调整参数名/签名：")
        print("  https://github.com/digitalcytometry/cytotrace2")
        print("常见问题：")
        print("  - 参数名可能为 snake_case 而非上述猜测")
        print("  - 返回值可能是 dict/DataFrame 而非 AnnData")
        print("  - 输入路径可能需要是目录而非单文件")
        print("=" * 60)
        _cytotrace2_success = False
        _ct2_result = None
else:
    if not RUN_CYTOTRACE2:
        print("RUN_CYTOTRACE2=False，跳过 CytoTRACE2 运行")
    elif not _cytotrace2_ready:
        print("CytoTRACE2 未就绪（包/权重不可用），跳过")
    _cytotrace2_success = False
    _ct2_result = None

In [ ]:
# === 读取 CytoTRACE2 结果，写回 adata.obs ===
# CytoTRACE2 返回对象的 .obs 中包含评分列（列名可能随版本变化）。
# 本 cell 尝试多种列名模式匹配，找到后对齐细胞索引写回 adata。
#
# 目标 obs 列：
#   - cytotrace2_potency_score    : 连续分化潜能分数（0-1，越高越未分化）
#   - cytotrace2_potency_category : 6 级分类（Totipotent/Pluripotent/.../Differentiated）
#   - cytotrace2_relative_order   : 相对排序（跨细胞排名）

if RUN_CYTOTRACE2 and _cytotrace2_ready and _cytotrace2_success:
    _result_loaded = False
    _result_df = None

    # --- 尝试方式 1：返回对象是 AnnData，从 .obs 提取 ---
    if not _result_loaded and hasattr(_ct2_result, "obs"):
        try:
            _result_df = _ct2_result.obs.copy()
            _result_df.index = _result_df.index.astype(str)
            _result_loaded = True
            print("从返回 AnnData.obs 读取结果")
        except Exception as _e:
            print(f"从 AnnData.obs 读取失败: {_e}")

    # --- 尝试方式 2：返回对象自身是 DataFrame ---
    if not _result_loaded and hasattr(_ct2_result, "columns"):
        try:
            _result_df = _ct2_result.copy()
            _result_df.index = _result_df.index.astype(str)
            _result_loaded = True
            print("从返回 DataFrame 读取结果")
        except Exception as _e:
            print(f"从 DataFrame 读取失败: {_e}")

    # --- 尝试方式 3：返回 dict，含 'df' 或 'results' key ---
    if not _result_loaded and isinstance(_ct2_result, dict):
        for _key in ["df", "results", "adata", "obs"]:
            if _key in _ct2_result:
                _candidate = _ct2_result[_key]
                if hasattr(_candidate, "obs"):
                    _result_df = _candidate.obs.copy()
                elif hasattr(_candidate, "columns"):
                    _result_df = _candidate.copy()
                else:
                    continue
                _result_df.index = _result_df.index.astype(str)
                _result_loaded = True
                print(f"从返回 dict['{_key}'] 读取结果")
                break

    # --- 尝试方式 4：从工作目录读取结果 CSV ---
    if not _result_loaded:
        _csv_patterns = [
            "expr_counts_cytotrace2_results.csv",
            "cytotrace2_results.csv",
            "cytotrace2_output.csv",
        ]
        for _fname in _csv_patterns:
            _csv_path = os.path.join(CYTOTRACE2_WORK_DIR, _fname)
            if os.path.exists(_csv_path):
                try:
                    _result_df = pd.read_csv(_csv_path, index_col=0)
                    _result_df.index = _result_df.index.astype(str)
                    _result_loaded = True
                    print(f"从文件读取结果: {_csv_path}")
                    break
                except Exception as _e:
                    print(f"读取 {_csv_path} 失败: {_e}")

    if not _result_loaded:
        print("无法读取 CytoTRACE2 结果：返回对象无法解析，且无结果 CSV")
        print(f"  返回对象类型: {type(_ct2_result).__name__}")
        print(f"  返回对象内容（前 200 字符）: {str(_ct2_result)[:200]}")
        _cytotrace2_success = False
    else:
        # --- 对齐细胞索引 ---
        _result_df = _result_df.reindex(adata.obs_names.astype(str))
        print(f"\n结果已对齐: {_result_df.shape[0]} 细胞")

        # --- 列名模式匹配 ---
        # CytoTRACE2 输出列名可能为 CamelCase 或 snake_case，按关键字匹配
        _col_map = {}  # {source_col: target_col}
        for _c in _result_df.columns:
            _cl = _c.lower().replace(" ", "_")
            # 连续分数（排除 preKNN 中间产物）
            if "cytotrace2" in _cl and "score" in _cl and "preknn" not in _cl:
                _col_map[_c] = "cytotrace2_potency_score"
            # 分类标签
            elif "cytotrace2" in _cl and ("potency" in _cl or "category" in _cl or "class" in _cl):
                _col_map[_c] = "cytotrace2_potency_category"
            # 相对排序
            elif "cytotrace2" in _cl and ("relative" in _cl or "order" in _cl or "rank" in _cl):
                _col_map[_c] = "cytotrace2_relative_order"

        if not _col_map:
            print("WARNING: 未能从结果列名中识别 CytoTRACE2 输出列")
            print(f"  可用列: {list(_result_df.columns)}")
            print("  请手动调整上方 _col_map 匹配逻辑")
        else:
            # 写回 adata.obs
            for _src, _dst in _col_map.items():
                adata.obs[_dst] = _result_df[_src].values
            print(f"\nCytoTRACE2 结果已写入 adata.obs:")
            for _dst in _col_map.values():
                if _dst in adata.obs.columns:
                    _s = adata.obs[_dst]
                    if _dst == "cytotrace2_potency_score":
                        _valid = _s.dropna()
                        print(f"  {_dst}: range [{_valid.min():.4f}, {_valid.max():.4f}]  "
                              f"mean={_valid.mean():.4f}  NA={_s.isna().sum()}")
                    elif _dst == "cytotrace2_potency_category":
                        print(f"  {_dst}: {_s.nunique()} 类  {sorted(_s.dropna().unique())}")
                    elif _dst == "cytotrace2_relative_order":
                        _valid = _s.dropna()
                        print(f"  {_dst}: range [{_valid.min():.0f}, {_valid.max():.0f}]")
else:
    if not RUN_CYTOTRACE2:
        print("RUN_CYTOTRACE2=False，跳过 CytoTRACE2，cytotrace2_potency_score 等列不会写入 adata.obs")
    elif not _cytotrace2_ready:
        print("CytoTRACE2 未运行（包/权重未就绪），cytotrace2_potency_score 等列不会写入 adata.obs")
    elif not _cytotrace2_success:
        print("CytoTRACE2 运行失败/结果读取失败，cytotrace2_potency_score 等列不会写入 adata.obs")

In [ ]:
# === 内存自检 ===
# 确保 adata.X 稀疏性/精度在流程中未被破坏。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

# 检查新增 obs 列
_new_cols = [
    "cytotrace2_potency_score",
    "cytotrace2_potency_category",
    "cytotrace2_relative_order",
]
for _c in _new_cols:
    if _c in adata.obs.columns:
        _v = adata.obs[_c]
        print(f"  {_c}: non-NA={_v.notna().sum()}/{len(_v)}")
    else:
        print(f"  {_c}: 未写入")

In [ ]:
# === 统一追踪字段 ===
# stage + version + upstream + status + 方法参数嵌套 dict
adata.uns["stage"] = "10d_pseudotime_cytotrace2"
adata.uns["version"] = "v1"
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"

# CytoTRACE2 方法细节——嵌套 dict 记录参数
_cytotrace2_uns = {
    "method": "cytotrace2",
    "species": CYTOTRACE2_SPECIES,
    "run_cytotrace2": RUN_CYTOTRACE2,
    "cytotrace2_ran": _cytotrace2_success if "_cytotrace2_success" in dir() else False,
    "epithelial_clusters": EPITHELIAL_CLUSTERS,
    "auto_subset_epithelial": AUTO_SUBSET_EPITHELIAL,
    "group_col": _group_col,
    "n_cells": adata.n_obs,
    "batch_size": CYTOTRACE2_BATCH_SIZE,
    "smooth_batch_size": CYTOTRACE2_SMOOTH_BATCH_SIZE,
    "max_workers": CYTOTRACE2_MAX_WORKERS,
    "disable_parallelization": CYTOTRACE2_DISABLE_PARALLEL,
    "work_dir": CYTOTRACE2_WORK_DIR,
}
adata.uns["10d_pseudotime_cytotrace2_v1"] = _cytotrace2_uns
print(f"追踪字段已写入: stage={adata.uns['stage']}  version={adata.uns['version']}  "
      f"status={adata.uns['status']}")
print(f"  cyotrace2_ran={_cytotrace2_uns['cytotrace2_ran']}  "
      f"n_cells={_cytotrace2_uns['n_cells']}")

In [ ]:
# === 写出 checkpoint ===
# 即使 CytoTRACE2 未运行（cytotrace2_ran=False），也写出 h5ad，
# 保证 pipeline 连贯——下游 notebook 可读取同一路径。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

# 如果 CytoTRACE2 未运行，额外确认标记已写入
if not (_cytotrace2_success if "_cytotrace2_success" in dir() else False):
    print("注意: CytoTRACE2 未运行，输出 h5ad 不含 potency 列（仅含 provenance 标记）")

In [ ]:
# === 释放内存 ===
del adata
# 清理可能的大对象引用
if "_ct2_result" in dir():
    del _ct2_result
if "_expr_df" in dir():
    del _expr_df
gc.collect()
print("内存已释放")